# Transactions - Transactions Pipeline Notebook

Dataset: `transactions_train.csv`

## 0 - Setup: SparkSession

In [1]:
import sys
sys.path.insert(0, '/app')  # Docker working directory

from src.spark_utils import create_spark_session

spark = create_spark_session('Transactions-Notebook')
print('SparkSession ready. Version:', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/13 20:19:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession ready. Version: 3.2.0


## Stage 3 - Extraction

### Step 1 - Define schema

In [2]:
from src.transactions.schema import TRANSACTIONS_SCHEMA

print('Schema definition:')
print(TRANSACTIONS_SCHEMA.simpleString())
print()
print('Fields:')
for f in TRANSACTIONS_SCHEMA.fields:
    print(f'  {f.name:<22} {str(f.dataType):<16} nullable={f.nullable}')

Schema definition:
struct<t_dat:date,customer_id:string,article_id:string,price:double,sales_channel_id:int>

Fields:
  t_dat                  DateType         nullable=True
  customer_id            StringType       nullable=True
  article_id             StringType       nullable=True
  price                  DoubleType       nullable=True
  sales_channel_id       IntegerType      nullable=True


### Step 2 - Load DataFrame with schema

In [3]:
from src.transactions.schema import load_transactions

RAW_PATH = '../../data/raw/transactions_train.csv'
raw_df = load_transactions(spark, RAW_PATH)

print('DataFrame created (lazy - no data read yet).')
print('Columns:', raw_df.columns)

DataFrame created (lazy - no data read yet).
Columns: ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']


### Step 3 - Verify load correctness

In [4]:
print('[Schema - confirmed types]')
raw_df.printSchema()

[Schema - confirmed types]
root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [5]:
row_count = raw_df.count()   # Action: reads the full CSV
print(f'Total rows : {row_count:,}')
print(f'Columns    : {len(raw_df.columns)}')

Total rows : 31,788,324
Columns    : 5


In [6]:
print('[Sample - first 10 rows]')
raw_df.show(10, truncate=False)

[Sample - first 10 rows]
+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|t_dat     |customer_id                                                     |article_id|price               |sales_channel_id|
+----------+----------------------------------------------------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|0663713001|0.050830508474576264|2               |
|2018-09-20|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|0541518023|0.03049152542372881 |2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|0505221004|0.01523728813559322 |2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|0685687003|0.016932203389830508|2               |
|2018-09-20|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|068568700

---
## Stage 4 - Preprocessing

### Step 1 - General statistical description

In [7]:
print('=== General Description ===')
print(f'Rows   : {raw_df.count():,}')
print(f'Columns: {len(raw_df.columns)} → {raw_df.columns}')
print()
print('[describe()]')
raw_df.describe().show(truncate=False)

=== General Description ===


Rows   : 31,788,324
Columns: 5 → ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']

[describe()]


+-------+----------------------------------------------------------------+--------------------+--------------------+-------------------+
|summary|customer_id                                                     |article_id          |price               |sales_channel_id   |
+-------+----------------------------------------------------------------+--------------------+--------------------+-------------------+
|count  |31788324                                                        |31788324            |31788324            |31788324           |
|mean   |null                                                            |6.962272191337916E8 |0.02782927385693629 |1.7040277430165869 |
|stddev |null                                                            |1.3344800348731518E8|0.019181128054793602|0.45647857193362845|
|min    |00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|0108775015          |1.694915254237288E-5|1                  |
|max    |ffffd9ac14e89946416d80e791d06470

### Step 2 - Numerical feature analysis

In [8]:
from pyspark.sql import functions as F

# Price percentiles
quantiles = raw_df.stat.approxQuantile('price', [0.0, 0.25, 0.5, 0.75, 1.0], 0.01)
print('[price stats]')
print(f'  min      : {quantiles[0]:.6f}')
print(f'  Q1       : {quantiles[1]:.6f}')
print(f'  median   : {quantiles[2]:.6f}')
print(f'  Q3       : {quantiles[3]:.6f}')
print(f'  max      : {quantiles[4]:.6f}')

agg = raw_df.select(
    F.mean('price').alias('mean'),
    F.stddev('price').alias('stddev')
).collect()[0]
print(f'  mean     : {agg["mean"]:.6f}')
print(f'  stddev   : {agg["stddev"]:.6f}')

[price stats]
  min      : 0.000017
  Q1       : 0.015797
  median   : 0.025407
  Q3       : 0.033881
  max      : 0.591525


  mean     : 0.027829
  stddev   : 0.019181


In [9]:
print('[sales_channel_id distribution]')
raw_df.groupBy('sales_channel_id').count().orderBy('sales_channel_id').show()

[sales_channel_id distribution]


+----------------+--------+
|sales_channel_id|   count|
+----------------+--------+
|               1| 9408462|
|               2|22379862|
+----------------+--------+



### Step 3 - Type casting & date parsing

In [10]:
from src.transactions.preprocessing import _step3_type_casting

typed_df = _step3_type_casting(raw_df)
typed_df.printSchema()
typed_df.select('t_dat', 'year', 'month', 'day_of_week', 'price', 'sales_channel_id').show(5)


--- Step 3 - type casting and date parsing ---
casts done, added year/month/day_of_week
root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

+----------+----+-----+-----------+--------------------+----------------+
|     t_dat|year|month|day_of_week|               price|sales_channel_id|
+----------+----+-----+-----------+--------------------+----------------+
|2018-09-20|2018|    9|    

### Step 4 - Feature informativeness analysis

In [11]:
total = typed_df.count()
print(f"{'Column':<22} {'Distinct':>10} {'Null %':>10}  Assessment")
print(f'{"─"*22} {"─"*10} {"─"*10}  {"─"*30}')
for col in typed_df.columns:
    distinct = typed_df.select(col).distinct().count()
    null_pct = typed_df.filter(F.col(col).isNull()).count() / total * 100
    if distinct == 1:
        verdict = 'DROP - zero variance'
    elif col in ('customer_id',):
        verdict = 'KEEP - join key'
    else:
        verdict = 'KEEP'
    print(f'{col:<22} {distinct:>10,} {null_pct:>9.2f}%  {verdict}')

Column                   Distinct     Null %  Assessment
────────────────────── ────────── ──────────  ──────────────────────────────


t_dat                         734      0.00%  KEEP


customer_id             1,362,281      0.00%  KEEP - join key


article_id                104,547      0.00%  KEEP


price                       9,857      0.00%  KEEP


sales_channel_id                2      0.00%  KEEP


year                            3      0.00%  KEEP


month                          12      0.00%  KEEP


day_of_week                     7      0.00%  KEEP


### Step 5 - Missing values & duplicate handling

In [12]:
from src.transactions.preprocessing import _step5_nulls_and_duplicates

# Null counts per column
typed_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in typed_df.columns
]).show(truncate=False)

+-----+-----------+----------+-----+----------------+----+-----+-----------+
|t_dat|customer_id|article_id|price|sales_channel_id|year|month|day_of_week|
+-----+-----------+----------+-----+----------------+----+-----+-----------+
|0    |0          |0         |0    |0               |0   |0    |0          |
+-----+-----------+----------+-----+----------------+----+-----+-----------+



In [13]:
clean_df = _step5_nulls_and_duplicates(typed_df)
print(f'Rows before: {typed_df.count():,}')
print(f'Rows after : {clean_df.count():,}')


--- Step 5 - nulls and duplicates ---


nulls per column:


+-----+-----------+----------+-----+----------------+----+-----+-----------+
|t_dat|customer_id|article_id|price|sales_channel_id|year|month|day_of_week|
+-----+-----------+----------+-----+----------------+----+-----+-----------+
|0    |0          |0         |0    |0               |0   |0    |0          |
+-----+-----------+----------+-----+----------------+----+-----+-----------+



rows dropped (nulls): 0



exact duplicate rows: 2,974,905 (kept - these are repeat purchases)


before: 31,788,324  after: 31,788,324  removed: 0


Rows before: 31,788,324


Rows after : 31,788,324


## Save Cleaned Data

In [14]:
OUT_PATH = '../../data/processed/transactions'
clean_df.write.mode('overwrite').parquet(OUT_PATH)
print(f'Saved to: {OUT_PATH}')
print('[Sample of saved data]')
spark.read.parquet(OUT_PATH).show(5)

Saved to: ../../data/processed/transactions
[Sample of saved data]
+----------+--------------------+----------+--------------------+----------------+----+-----+-----------+
|     t_dat|         customer_id|article_id|               price|sales_channel_id|year|month|day_of_week|
+----------+--------------------+----------+--------------------+----------------+----+-----+-----------+
|2019-11-29|aaa78c87aacba903d...|0706016003|0.025288135593220337|               2|2019|   11|          6|
|2019-11-29|aaa78c87aacba903d...|0706016001|0.025288135593220337|               2|2019|   11|          6|
|2019-11-29|aaa78c87aacba903d...|0682236013|0.018983050847457626|               2|2019|   11|          6|
|2019-11-29|aaa78c87aacba903d...|0706016016|0.025288135593220337|               2|2019|   11|          6|
|2019-11-29|aaa7a0483dd5b9e39...|0783335003|0.020322033898305086|               2|2019|   11|          6|
+----------+--------------------+----------+--------------------+----------------+---

In [15]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.
